# kazeval on Kaggle — canonical GPU runs (compute policy 2026-07-04)

One-time Kaggle setup:
1. **Settings → Accelerator**: GPU T4 x2 or P100 (both 16 GB, fp16 — no bf16).
2. **Settings → Internet**: ON (needs a phone-verified account).
3. **Add-ons → Secrets**: add `HF_TOKEN` = a HuggingFace token of an account that has
   accepted the conditions of the gated datasets/models used below
   ([issai/kazqad-retrieval](https://huggingface.co/datasets/issai/kazqad-retrieval),
   and [inceptionai/Llama-3.1-Sherkala-8B-Chat](https://huggingface.co/inceptionai/Llama-3.1-Sherkala-8B-Chat) for the KazMMLU ceiling run).

The repo is cloned from GitHub `main` — **push before launching**, Kaggle sees only what is pushed.
Free quota: 30 GPU-h/week, sessions ≤ 12 h — long runs must checkpoint/finish inside one session.


In [ ]:
# Install from PRE-BUILT wheels shipped as a Kaggle dataset (kekeront/qymyzlm-wheels).
# Same fix as the training kernel (v3 post-mortem): a run-time GitHub clone executes
# whatever main happens to hold, not the bytes verified locally. Wheels are built
# from a fresh clone and content-verified against git ls-files before upload.
import glob
import pathlib
import subprocess
import sys

WHEEL_DIR = "/kaggle/input/qymyzlm-wheels"
print(open(f"{WHEEL_DIR}/MANIFEST.txt").read().strip())
kazeval_whl = sorted(glob.glob(f"{WHEEL_DIR}/kazeval-*.whl"))
assert kazeval_whl, f"kazeval wheel missing from {WHEEL_DIR}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", kazeval_whl[-1]])
subprocess.check_call([sys.executable, "-c", "import kazeval.run_retrieval"])
print("INSTALL_OK: kazeval importable")

from huggingface_hub import login


def hf_token():
    """Kaggle secret HF_TOKEN (interactive runs) or an attached private
    token dataset with hf_token.txt (kernels pushed via the Kaggle API,
    which cannot access UserSecretsClient secrets)."""
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hits = sorted(pathlib.Path("/kaggle/input").glob("**/hf_token.txt"))
        if not hits:
            raise RuntimeError(
                "no HF_TOKEN: add a Kaggle secret or attach the private token dataset"
            ) from None
        return hits[0].read_text().strip()


login(hf_token())


In [ ]:
# Heartbeat side-channel: Kaggle's API exposes only RUNNING/COMPLETE — no elapsed
# time, no mid-run logs (v7 post-mortem: a hung run was indistinguishable from a
# healthy one for 10h). beat() uploads a tiny JSON to a PRIVATE HF dataset repo;
# poll locally with evallab/kaggle/heartbeat_watch.py. Never fatal.
import json
import time

HB_REPO = "kekeront/qymyzlm-run-heartbeat"
HB_RUN = "kazeval-leaderboard-v0"
_HB_T0 = time.time()


def beat(stage, **info):
    payload = {
        "run": HB_RUN,
        "stage": stage,
        "elapsed_s": round(time.time() - _HB_T0),
        "ts_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        **info,
    }
    try:
        from huggingface_hub import HfApi

        HfApi().upload_file(
            path_or_fileobj=json.dumps(payload, ensure_ascii=False).encode(),
            path_in_repo="heartbeat.json",
            repo_id=HB_REPO,
            repo_type="dataset",
            commit_message=f"beat: {HB_RUN} {stage}",
        )
    except Exception as exc:  # noqa: BLE001 — heartbeat is never fatal
        print(f"[heartbeat] non-fatal: {exc}")
    print(f"[heartbeat] {payload['elapsed_s']}s {stage} {info if info else ''}", flush=True)


beat("session_start")


## Run 1 — leaderboard-v0 embedding baselines (KazQADRetrieval)

Full-corpus retrieval only (1,929 KazQAD test queries vs the 825,309-passage Kazakh
Wikipedia corpus) for **five off-the-shelf encoders** through the pinned
`KazQADRetrieval` protocol — this is the run that moves the measured-record count
from 1 to 6 and pays down the apparatus freeze. **Keystone first:** re-measure
`intfloat/multilingual-e5-large` (its KazQADRetrieval *measured* record is the
canonical planka), then `BAAI/bge-m3`, `Qwen/Qwen3-Embedding-0.6B`,
`sentence-transformers/LaBSE`, and
`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`.

Each model runs in its own `kazeval.run_retrieval` subprocess on both T4s
(`--device cuda:0,cuda:1`, fp16), writing one committed-style JSON record per model
into `/kaggle/working/results/`. Reranking is NOT part of v0 — retrieval only.

In [ ]:
# Leaderboard v0 (v9) — 4 encoders on TWO SINGLE-GPU LANES in parallel.
#
# v7 POST-MORTEM (2026-07-05): --device cuda:0,cuda:1 routes through sentence-
# transformers' multi-process pool (main + 2 CUDA workers over IPC). v7 hung in it
# for ~10h with no timeout and was cancelled with zero records. This runner removes
# that failure mode STRUCTURALLY: each model runs in its own subprocess pinned to
# ONE GPU (no pool, no IPC), and the two T4s run different models concurrently:
#
#   cuda:0: BGE-M3 (~3-4h)      -> MiniLM-L12 (~0.5h)
#   cuda:1: Qwen3-Emb-0.6B (~4h) -> LaBSE (~1h)
#
# Wall-clock ~= max(lane) ~= 5h; worst case = 2 x 4h timeouts per lane < 12h cap.
# mE5-large keystone is NOT re-run: its measured record (kernel v6) is committed
# in evallab/results/ — re-measuring would cost ~3.5h for zero new information.
#
# CORRECTNESS GUARD (unchanged from v8): a model missing from mteb's registry
# falls back to a bare SentenceTransformer WITHOUT prompts -> wrong-but-plausible
# number. run_retrieval only LOGS that, so we tee+capture output and quarantine
# (rename .INVALID) any record whose run logged the fallback mark.

import os
import shlex
import signal
import subprocess
import sys
import threading
import time
from pathlib import Path

RESULTS = Path("/kaggle/working/results")
RESULTS.mkdir(parents=True, exist_ok=True)

LANES = {
    "cuda:0": [
        "BAAI/bge-m3",
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    ],
    "cuda:1": [
        "Qwen/Qwen3-Embedding-0.6B",
        "sentence-transformers/LaBSE",
    ],
}
BATCH = {  # fp16 on a 16 GB T4, per-model headroom-tuned
    "BAAI/bge-m3": 128,
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": 256,
    "Qwen/Qwen3-Embedding-0.6B": 64,
    "sentence-transformers/LaBSE": 128,
}
DTYPE = "float16"  # T4: no bf16; run_retrieval hard-fails a bad cast
PER_MODEL_TIMEOUT = 4 * 3600
LANE_STAGGER_S = 300  # lane 2 waits for lane 1 to win the gated-dataset download race
FALLBACK_MARK = "not in mteb's model registry"

_lock = threading.Lock()  # serializes snapshot/quarantine/prints across lanes


def snapshot():
    return {p: p.stat().st_mtime for p in RESULTS.glob("*.json")}


def quarantine(new_paths, reason):
    for p in new_paths:
        bad = p.with_suffix(p.suffix + ".INVALID")
        p.rename(bad)
        print(f"!!! QUARANTINED ({reason}): {bad.name}", flush=True)


def run_one(model, device):
    bs = BATCH.get(model, 64)
    cmd = [
        sys.executable, "-m", "kazeval.run_retrieval",
        "--model", model,
        "--tasks", "KazQADRetrieval",
        "--batch-size", str(bs),
        "--device", device,  # SINGLE device: no multi-process pool, no IPC
        "--dtype", DTYPE,
        "--output-dir", str(RESULTS),
    ]
    with _lock:
        print(f"\n>>> [{device}] {model} (batch={bs})\n>>> {shlex.join(cmd)}", flush=True)
        before = snapshot()
    beat("eval_start", model=model, lane=device)
    t0 = time.time()
    buf = []
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        start_new_session=True,  # own process group -> timeout SIGKILLs everything
    )

    def pump():
        for line in proc.stdout:
            buf.append(line)
            with _lock:
                print(f"[{device}] {line}", end="", flush=True)

    th = threading.Thread(target=pump, daemon=True)
    th.start()
    try:
        rc = proc.wait(timeout=PER_MODEL_TIMEOUT)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()
        th.join(timeout=60)
        with _lock:
            quarantine([p for p in snapshot() if p not in before], "timeout")
        print(f"TIMEOUT after {PER_MODEL_TIMEOUT}s: [{device}] {model}", flush=True)
        beat("eval_timeout", model=model, lane=device)
        return
    th.join(timeout=60)
    dur_m = (time.time() - t0) / 60
    with _lock:
        new = [p for p in snapshot() if p not in before]
        if rc != 0:
            quarantine(new, f"exit {rc}")
            beat("eval_failed", model=model, lane=device, exit_code=rc)
        elif FALLBACK_MARK in "".join(buf):
            quarantine(new, "registry fallback -> prompts wrong")
            beat("eval_invalid", model=model, lane=device)
        else:
            print(f"OK [{device}] {model} in {dur_m:.0f} min -> {[p.name for p in new]}", flush=True)
            beat("eval_done", model=model, lane=device, minutes=round(dur_m))


def lane_worker(device, models, delay_s):
    if delay_s:
        time.sleep(delay_s)
    for m in models:
        run_one(m, device)


threads = [
    threading.Thread(target=lane_worker, args=(dev, models, i * LANE_STAGGER_S))
    for i, (dev, models) in enumerate(LANES.items())
]
for t in threads:
    t.start()
for t in threads:
    t.join()

n = len(list(RESULTS.glob("*.json")))
beat("all_done", records=n)
print(f"\nall lanes done: {n} record(s) on {RESULTS}", flush=True)


## Run 2 (optional) — KazMMLU 3-shot, Sherkala-Chat-8B in 4-bit (W5: same-protocol ceiling)

8B in 4-bit ≈ 5.5 GB weights — fits a single T4/P100. Uncomment to run.


In [ ]:
# %pip install -q bitsandbytes
# !python -m kazeval.run_kazmmlu \
#     --model inceptionai/Llama-3.1-Sherkala-8B-Chat \
#     --model-args dtype=float16,load_in_4bit=True \
#     --batch-size auto \
#     --output-dir /kaggle/working/results


## Collect results

Download every JSON under `/kaggle/working/results/` (Output tab), drop the files into
`evallab/results/` in the repo, then locally:

```bash
python -m kazeval.leaderboard   # re-renders the README leaderboard
```


In [ ]:
# Collect — validate every record on /kaggle/working/results, preview the retrieval
# leaderboard, and dump full JSON for download. Validation via kazeval.results.load_record
# catches a half-written file from a SIGKILL'd/timed-out process before it is committed.

from pathlib import Path

from kazeval.results import load_record

RESULTS = Path("/kaggle/working/results")
paths = sorted(RESULTS.glob("*.json"))
print(f"{len(paths)} record file(s) under {RESULTS}\n")

rows, bad = [], []
for p in paths:
    try:
        rows.append(load_record(p))  # schema-validates; rejects malformed/partial JSON
    except Exception as exc:
        bad.append((p.name, str(exc)))

# KazQADRetrieval leaderboard preview (main_score = ndcg_at_10, higher is better).
ret = sorted(
    (r for r in rows if r.task == "KazQADRetrieval"),
    key=lambda r: r.metrics.get("ndcg_at_10", -1.0),
    reverse=True,
)
if ret:
    w = max(len(r.model) for r in ret)
    print(f"{'model':<{w}}  prov      ndcg@10  mrr@10  recall@100")
    for r in ret:
        m = r.metrics
        nan = float("nan")
        print(
            f"{r.model:<{w}}  {r.provenance:<8}  "
            f"{m.get('ndcg_at_10', nan):7.4f}  "
            f"{m.get('mrr_at_10', nan):6.4f}  "
            f"{m.get('recall_at_100', nan):7.4f}"
        )
    print()

# Full JSON so it can be copied out even if the Output download is missed.
for p in paths:
    print(f"----- {p.name} -----")
    print(p.read_text())

if bad:
    print("\n!!! INVALID record files (do NOT commit — re-run these models):")
    for name, err in bad:
        print(f"  {name}: {err}")
else:
    print(
        "\nAll record files validate. Download them (Output tab) into evallab/results/, "
        "then locally: python -m kazeval.leaderboard"
    )